# Phase 2 — Pairwise Co-mutation Matrix

**Purpose**: for each cancer type, test every pair of frequently-altered
genes for co-occurrence (altered together more than chance) or mutual
exclusivity (altered together less than chance), using a DISCOVER-style
rate-adjusted test (Section 4) -- not Fisher's exact test, which turned out
to badly overstate co-occurrence once mutation-burden skew was checked
directly (see Section 4 markdown for the evidence and the fix). This turns
the flat alteration list from Phase 1 into the actual co-mutation matrix
your supervisor asked for.

**On terminology**: "co-mutation" here means co-occurring **genomic
alteration** broadly, not point mutations alone -- every "altered" flag in
this notebook already combines both mutations and CNVs (confirmed: some of
the strongest patterns found, e.g. the 11q13/8p12 amplicon pair, are almost
entirely CNV-driven, not mutation-driven at all). See Section 6a below for
the per-pair mutation/CNV mechanism breakdown this implies.

**CNV calls used**: deep amplification / deep deletion only, with
direction-inconsistent bystander calls already removed by Phase 1 (Section
4b there). Calls on genes OncoKB doesn't curate are kept.

**Inputs** (from `data/processed/`, built by Phase 1):
- `alterations_long.parquet`
- `panel_gene_coverage.parquet`
- `clinical_tidy.parquet`
- `consensus_genes_per_cancer_type.parquet` -- a loose per-cancer-type
  candidate pool (>=50 tested patients, no coverage-fraction requirement --
  see Phase 1 Section 8b for why the old 80%-coverage version was replaced).
  Reliability is enforced per-*pair* here instead (Section 4's joint-coverage
  gate), not per-gene by this list.
- `consensus_genes_global.parquet` -- the smaller, pooled-cohort gene list
  (~190 genes) safe to compare *across* cancer types; used for the second run

**Outputs**:
- `data/processed/comutation_pairs.parquet` -- main run, per-cancer-type
  gene lists, one row per (cancer type, gene pair) tested against the
  rate-adjusted null, annotated with a MUT/CNV mechanism breakdown and
  genomic-proximity flag (Section 6a)
- `data/processed/comutation_pairs_global_genes.parquet` -- second run,
  same 190-gene list for every cancer type, for direct cross-cancer-type
  comparison
- `data/processed/mechanism_pairs.parquet` -- each pair tested four ways
  (SNV/CNV on each side), so a pair's signal can be attributed to a
  specific alteration-type combination (Section 6b)
- `data/processed/pan_cancer_pairs.parquet` -- Aim 1's "across cancers"
  half: which gene pairs recur as significant, with a consistent
  direction, across multiple cancer types (Section 7)

This notebook is self-contained, like Phase 1 -- no separate script
dependency. See `PLAN.md` Phase 2 for the full methodology writeup.

## Phase 2 data flow, at a glance

```text
alterations_long.parquet + clinical_tidy.parquet
              |
              |  keep only cancer types with >= 100 samples
              v
     69 qualifying cancer types (of 112 total); 59 keep >= 100 samples
     once CNA-untested samples are excluded (the main run's cohort)


for EACH qualifying cancer type:

  alterations_long.parquet (this cancer type's rows)
              |
              |  keep only genes altered in >= max(5, 3% of samples) AND
              |  in Phase 1's loose per-cancer-type candidate pool
              |  (>=50 tested patients, no coverage-fraction requirement
              |  -- see Phase 1 Section 8b)
              v
     qualifying_genes  (varies widely per cancer type -- frequency is
     the binding constraint, not the candidate pool's size)


  panel_gene_coverage.parquet  +  cna_gene_panel_coverage.parquet
  + clinical_tidy.parquet (this cancer type)
              |
              |  TWO masks, not one (Phase 1 Section 2b): the mutation target
              |  list and where copy-number calls actually exist disagree for
              |  16.6% of (panel, gene) combos
              v
     mut_cov  (samples x genes, "mutation was testable here")
     cna_cov  (samples x genes, "copy number was testable here")
              |
              |  main run scores altered = MUT or CNV, so a "not altered"
              |  label is only trustworthy when BOTH were observable
              v
     tested = mut_cov AND cna_cov
     (mechanism tests instead use whichever mask matches the side tested)


  alteration_matrix + coverage_matrix
              |
              |  fit_gene_sample_rates: iterative proportional fitting of
              |  a per-gene x per-patient Poisson rate table (Section 4a),
              |  restricted to tested cells -- corrects for the mutation-
              |  burden skew that made Fisher's exact test overstate
              |  co-occurrence (Section 4 markdown)
              v
     pi  (samples x qualifying_genes, rate-adjusted P(altered))


  alteration_matrix + coverage_matrix + pi
              |
              |  for every gene pair (A, B):
              |    joint-coverage gate -- restrict to samples where BOTH
              |    A and B were tested, require that's >=50% of this
              |    cancer type's own patients (not just an absolute floor)
              |    and >=10 patients show BOTH altered (MIN_BOTH)
              |    -> test observed co-occurrence against the
              |    Poisson-Binomial null implied by pi (Section 4)
              v
     one row per gene pair: fold enrichment + p-value


all gene-pair rows, across all 59 main-run cancer types
              |
              |  Benjamini-Hochberg FDR correction, applied separately
              |  within each cancer type (each cancer type ran its own
              |  batch of tests, so each gets its own correction)
              v
     comutation_pairs.parquet  (25,609 rows, 12,711 significant at q<0.05)
     Purpose: ONE table of every gene pair tested, per cancer type, with
     a q-value telling us which enrichments/exclusivities are trustworthy
```

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 50)

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

DATA_DIR = Path.cwd().parent / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

MIN_SAMPLES_PER_CANCER_TYPE = 100
MIN_GENE_ALT_FRACTION = 0.03
MIN_GENE_ALT_ABS = 5
# Per-pair joint-coverage gate (replaces the old single-gene 80%-of-population
# prefilter from Phase 1's per-cancer-type gene list -- see Phase 1 Section 8b).
# A pair is only tested if enough of THIS cancer type's own patients were
# tested for BOTH genes together, not each gene's marginal coverage checked
# in isolation. Fixes cases like Melanoma, where BRAF/NRAS/KRAS are each
# tested in 97-100% of patients individually, but many otherwise-solid genes
# sit at 60-79% specifically within Melanoma (its 10,203 patients are split
# across 87 different panels) and were being dropped before ever reaching a
# pairwise test, despite having plenty of joint-tested patients in absolute
# terms.
JOINT_COVERAGE_FRACTION = 0.5
# A pair also needs at least this many patients with BOTH genes altered
# (not just enough patients tested) -- mirrors MECH_MIN_BOTH below, applied
# here to the main run too. At a floor of 5, some earlier significant pairs
# rested on single-digit co-occurrence counts with unstable, sometimes
# infinite odds ratios; 10 removes that tail (see Section 4 markdown).
MIN_BOTH = 10


## 1. Load Phase 1 outputs

In [ ]:
# Phase 1 already dropped direction-inconsistent (bystander) CNV calls, so
# this is the driver-focused table as-is. Kept CNV rows still carry
# CNV_Driver_Status = Driver_Consistent | Unannotated.
alterations = pd.read_parquet(PROCESSED_DIR / "alterations_long.parquet")
panel_coverage = pd.read_parquet(PROCESSED_DIR / "panel_gene_coverage.parquet")
# Mutation target list vs. where copy-number calls actually exist -- these are
# different things and disagree for 16.6% of (panel, gene) combos (Phase 1 Section 2b).
cna_panel_coverage = pd.read_parquet(PROCESSED_DIR / "cna_gene_panel_coverage.parquet")
clinical = pd.read_parquet(PROCESSED_DIR / "clinical_tidy.parquet")

print(alterations.shape, panel_coverage.shape, cna_panel_coverage.shape, clinical.shape)
print(alterations.loc[alterations["Alteration_Type"] == "CNV", "CNV_Driver_Status"].value_counts().to_string())

## 2. Select qualifying cancer types (>= 100 samples)

In [ ]:
cancer_type_counts = clinical["CANCER_TYPE"].value_counts()
qualifying_cancer_types = cancer_type_counts[cancer_type_counts >= MIN_SAMPLES_PER_CANCER_TYPE].index.tolist()
print(f"{len(qualifying_cancer_types)} of {len(cancer_type_counts)} cancer types qualify")
cancer_type_counts[cancer_type_counts >= MIN_SAMPLES_PER_CANCER_TYPE].head(10)

## 3. Per-cancer-type matrices

For a given cancer type: pick genes altered often enough to test, then
build two aligned samples x genes matrices -- one for "was this gene
altered" and one for "was this gene even tested" (the panel coverage
mask). Every pairwise test below restricts to samples where BOTH genes
in the pair were tested, using the second matrix.

**Two testability masks, not one.** `build_matrices` returns `mut_cov` and
`cna_cov` separately (Phase 1 Section 2b): `panel_gene_coverage.parquet`
describes each panel's *mutation* target list, while
`cna_gene_panel_coverage.parquet` records where copy-number calls actually
exist. They disagree for 16.6% of (panel, gene) combinations -- covering
~199,000 (sample, gene) cells that a single blended mask would score as
"tested, no copy-number change" when copy number was never assessed there
at all. The main run intersects them (a "not altered" label needs both
mechanisms observable); the mechanism-specific tests pick the mask matching
each side.

In [ ]:
def get_qualifying_genes(cancer_type: str, sub_alt: pd.DataFrame, n_samples: int, allowed_genes: set[str]) -> list[str]:
    """
    A gene qualifies if it's BOTH frequently altered AND in the
    caller-supplied allowed set -- either that cancer type's own
    candidate pool, or the global list, depending on which run this is
    (see Section 6).
    """
    gene_sample_counts = sub_alt.groupby("Hugo_Symbol")["Sample_ID"].nunique()
    min_count = max(MIN_GENE_ALT_ABS, MIN_GENE_ALT_FRACTION * n_samples)
    freq_qualified = set(gene_sample_counts[gene_sample_counts >= min_count].index)
    return sorted(freq_qualified & allowed_genes)


def _coverage_mask(cov_table: pd.DataFrame, ct_samples: pd.DataFrame, genes: list[str]) -> np.ndarray:
    """samples x genes bool: does this sample's panel cover this gene, per cov_table."""
    sub = cov_table[cov_table["Hugo_Symbol"].isin(genes)]
    panel_matrix = (sub.assign(covered=True)
                    .pivot_table(index="SEQ_ASSAY_ID", columns="Hugo_Symbol", values="covered", aggfunc="first")
                    .reindex(columns=genes).fillna(False).astype(bool))
    m = panel_matrix.reindex(ct_samples["SEQ_ASSAY_ID"]).fillna(False).astype(bool)
    m.index = ct_samples.index
    return m.to_numpy(dtype=bool)


def build_matrices(cancer_type: str, allowed_genes: set[str],
                   alt_source: pd.DataFrame | None = None,
                   require_cna_tested: bool = False):
    """
    Returns (alt, mut_cov, cna_cov, genes, n_samples) as numpy bool arrays.

    Two separate testability masks, NOT one (see Phase 1 Section 2b):
      mut_cov -- panel_gene_coverage.parquet, each panel's mutation target list
      cna_cov -- cna_gene_panel_coverage.parquet, where copy-number calls
                 actually exist, read from data_CNA.txt itself
    They disagree for 16.6% of (panel, gene) combinations. Using the mutation
    list for both silently scores ~199k (sample, gene) cells as "tested, no
    copy-number change" when copy number was never assessed there.
    """
    ct_samples = clinical.loc[clinical["CANCER_TYPE"] == cancer_type,
                              ["SAMPLE_ID", "SEQ_ASSAY_ID", "CNA_TESTED"]]
    ct_samples = ct_samples.drop_duplicates(subset="SAMPLE_ID").set_index("SAMPLE_ID")
    if require_cna_tested:
        # "Altered" has to mean the same thing for every sample in the table.
        # A sample with no copy-number data can only ever show a mutation, so
        # scoring it "not altered" misclassifies the outcome -- the same
        # tested-vs-never-tested error Phase 1 exists to prevent, applied to
        # CNVs instead of mutations.
        ct_samples = ct_samples[ct_samples["CNA_TESTED"]]
    n_samples = len(ct_samples)
    if n_samples < MIN_SAMPLES_PER_CANCER_TYPE:
        return None

    src = alterations if alt_source is None else alt_source
    sub_alt = src.loc[src["CANCER_TYPE"] == cancer_type, ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
    sub_alt = sub_alt[sub_alt["Sample_ID"].isin(ct_samples.index)]
    genes = get_qualifying_genes(cancer_type, sub_alt, n_samples, allowed_genes)
    if len(genes) < 2:
        return None

    # samples x genes, True where altered (mutation OR copy-number change)
    alt = (sub_alt[sub_alt["Hugo_Symbol"].isin(genes)]
           .assign(altered=True)
           .pivot(index="Sample_ID", columns="Hugo_Symbol", values="altered")
           .reindex(index=ct_samples.index, columns=genes)
           .fillna(False).astype(bool)).to_numpy()

    mut_cov = _coverage_mask(panel_coverage, ct_samples, genes)
    cna_cov = _coverage_mask(cna_panel_coverage, ct_samples, genes)
    return alt, mut_cov, cna_cov, genes, n_samples

## 4. Pairwise test: rate-adjusted (DISCOVER-style), not Fisher's exact test

**Why not Fisher's exact test.** Jason's question after seeing early results:
*"Are you using a Fisher's exact test to evaluate significant co-occurrence
or mutual exclusivity? Half being significant sounds quite high as I would
imagine many gene pairs have very few samples. Also you should apply
multiple testing correction."*

BH-FDR correction was already being applied (Section 5) -- that part of the
concern was already handled. But checking the "half seems high" half of the
question turned up a real, specific cause, not a vague one:

- Per-sample alteration burden is wildly skewed within a cancer type (e.g.
  Melanoma: median 6 genes altered per patient, 90th pct 26, max 494).
  Fisher's exact test assumes every patient has the same baseline chance of
  any given gene being altered -- that assumption is badly wrong here.
- Checked directly: of *all* significant pairs under Fisher's test, 90-94%
  were "co-occurring" and only 6-9% "exclusive" -- and the cancer types
  with the most lopsided split were exactly the ones already flagged
  elsewhere as inherently hypermutated (Skin Cancer Non-Melanoma, Endometrial,
  Melanoma). A burden confound inflates apparent co-occurrence specifically
  (a hypermutated patient's dozens of alterations make *every* gene pair on
  their panel look "co-occurring," purely from volume) while leaving
  genuine exclusivity comparatively untouched -- so a heavy co-occurring
  skew is the fingerprint of exactly this confound, not a generic red flag.

**The fix**: a DISCOVER-style test (Canisius et al. 2016, *Genome Biology*,
*"A novel independence test for somatic alterations in cancer shows that
biology drives mutual exclusivity but chance explains most co-occurrence"*
-- the title is almost exactly this section's finding). Instead of assuming
every patient has equal odds of a gene being altered, fit a rate model per
gene and per patient (Section 4a), then test each pair's observed
co-occurrence against the Poisson-Binomial distribution implied by *those*
patient- and gene-specific rates, instead of a single pooled probability.

**Controlled before/after** (identical gene lists, identical joint-coverage
gate, identical `n_both_altered >= 10` floor -- only the test statistic
differs):

| | Fisher's exact | DISCOVER-style |
|---|---|---|
| Significant (q<0.05) | 18,528 / 25,609 (72.3%) | 12,711 / 25,609 (49.6%) |
| Co-occurring : Exclusive | 94.4% : 5.6% | 82.9% : 17.1% |

Known biology survives the switch and is if anything clearer: *KRAS*/*EGFR*
in NSCLC goes from "significant" under Fisher to 173 observed vs. **1,425.5
expected** co-mutations under the rate-adjusted null (q≈0); the 11q13/8p12
breast-specific co-amplification (Section 10) is still 8/8 significant,
still breast-only.

### 4a. Rate model: fit each gene's and each patient's alteration rate

Iterative proportional fitting (the RAS/"raking" algorithm, a standard
categorical-data-analysis technique) on a Poisson rate table, restricted to
TESTED cells only -- a sample/gene pair the panel never covered contributes
to neither margin. `pi_ij = 1 - exp(-lam_ij)` keeps every fitted probability
naturally in [0, 1) with no ad-hoc capping. Converges to near-exact recovery
of both margins (checked: max absolute error < 1e-6 after ~10 iterations on
every cancer type tried).

In [ ]:
def fit_gene_sample_rates(alt_matrix: np.ndarray, tested_matrix: np.ndarray,
                            n_iter: int = 25) -> np.ndarray:
    """
    alt_matrix, tested_matrix: (n_samples, n_genes) boolean arrays.
    Returns lam (n_samples, n_genes): a Poisson rate table fit so that,
    summed over TESTED cells only, row sums (each patient's true alteration
    count) and column sums (each gene's true alteration count) are
    recovered almost exactly.
    """
    alt = alt_matrix.astype(np.float64)
    tested = tested_matrix.astype(np.float64)
    row_target = (alt * tested).sum(axis=1)
    col_target = (alt * tested).sum(axis=0)

    n_samples, n_genes = alt.shape
    gene_rate = np.full(n_genes, 0.1)
    sample_rate = np.full(n_samples, 1.0)

    for _ in range(n_iter):
        # No "leave unchanged if target is zero" special-casing: a patient
        # with zero alterations among these genes (or a gene with zero
        # alterations among these patients) must have its rate driven toward
        # zero too, or the fit never actually converges.
        lam = np.outer(sample_rate, gene_rate) * tested
        col_sum = lam.sum(axis=0)
        col_sum[col_sum < 1e-12] = 1e-12
        gene_rate = gene_rate * (col_target / col_sum)
        gene_rate = np.clip(gene_rate, 1e-10, None)

        lam = np.outer(sample_rate, gene_rate) * tested
        row_sum = lam.sum(axis=1)
        row_sum[row_sum < 1e-12] = 1e-12
        sample_rate = sample_rate * (row_target / row_sum)
        sample_rate = np.clip(sample_rate, 1e-10, None)

    return np.outer(sample_rate, gene_rate) * tested


def poisson_binomial_tail(probs: np.ndarray, k: float, upper: bool, exact_threshold: int = 1200) -> float:
    """P(X >= k) if upper else P(X <= k), for X ~ PoissonBinomial(probs).
    Exact via DFT (Fernandez & Williams 2010) for n <= exact_threshold;
    Volkova's (1996) refined normal approximation above that (the two agree
    to <1% on a shared test case -- checked directly)."""
    probs = np.clip(np.asarray(probs, dtype=np.float64), 1e-12, 1 - 1e-12)
    n = len(probs)
    if n == 0:
        return 1.0
    mu = probs.sum()
    var = (probs * (1 - probs)).sum()
    if var <= 1e-10:
        return 0.0 if (upper and k <= mu) or (not upper and k >= mu) else 1.0

    if n <= exact_threshold:
        C = np.exp(2j * np.pi / (n + 1))
        ls = np.arange(n + 1)
        chi = np.ones(n + 1, dtype=np.complex128)
        chunk = 300
        for start in range(0, n, chunk):
            p_chunk = probs[start:start + chunk]
            terms = 1 + np.outer((C ** ls) - 1, p_chunk)
            chi *= terms.prod(axis=1)
        ks = np.arange(n + 1)
        Cinv_pow = C ** (-np.outer(ks, ls))
        pmf = np.real(Cinv_pow @ chi) / (n + 1)
        pmf = np.clip(pmf, 0, None)
        s = pmf.sum()
        if s > 0:
            pmf /= s
        return float(pmf[int(np.ceil(k)):].sum()) if upper else float(pmf[:int(np.floor(k)) + 1].sum())

    sigma = np.sqrt(var)
    gamma = (probs * (1 - probs) * (1 - 2 * probs)).sum() / (sigma ** 3)
    if upper:
        x = (k - 0.5 - mu) / sigma
        p = 1 - norm.cdf(x) - gamma / 6 * (1 - x ** 2) * norm.pdf(x)
    else:
        x = (k + 0.5 - mu) / sigma
        p = norm.cdf(x) + gamma / 6 * (1 - x ** 2) * norm.pdf(x)
    return float(np.clip(p, 0, 1))


def discover_pair_test(pi_a: np.ndarray, pi_b: np.ndarray, observed_both: int) -> tuple:
    """Two-sided p-value + direction for the observed co-occurrence count,
    against the rate-adjusted independence null (product of each patient's
    fitted per-gene probabilities, summed into a Poisson-Binomial null)."""
    joint_probs = pi_a * pi_b
    expected = joint_probs.sum()
    if observed_both >= expected:
        p_one_sided = poisson_binomial_tail(joint_probs, observed_both, upper=True)
        direction = "co-occurring"
    else:
        p_one_sided = poisson_binomial_tail(joint_probs, observed_both, upper=False)
        direction = "exclusive"
    return min(1.0, 2 * p_one_sided), direction, expected

In [ ]:
def test_cancer_type(cancer_type: str, allowed_genes: set[str],
                     alt_source: pd.DataFrame | None = None,
                     require_cna_tested: bool = False) -> pd.DataFrame:
    result = build_matrices(cancer_type, allowed_genes, alt_source, require_cna_tested)
    if result is None:
        return pd.DataFrame()
    alt, mut_cov, cna_cov, genes, n_ct_samples = result

    # "altered" = mutation OR copy-number change, so a "not altered" label is
    # only trustworthy when BOTH mechanisms were observable for that gene in
    # that sample -- hence the intersection, not the mutation mask alone.
    tested = mut_cov & cna_cov

    lam = fit_gene_sample_rates(alt, tested)
    pi = 1 - np.exp(-lam)

    rows = []
    for gene_a, gene_b in itertools.combinations(range(len(genes)), 2):
        joint_tested = tested[:, gene_a] & tested[:, gene_b]
        n_tested = int(joint_tested.sum())
        # Joint-coverage gate: enough of THIS cancer type's own patients were
        # tested for BOTH genes, not each gene's marginal coverage checked in
        # isolation (see Phase 1 Section 8b).
        if n_tested < MIN_GENE_ALT_ABS or n_tested / n_ct_samples < JOINT_COVERAGE_FRACTION:
            continue

        both = int((alt[:, gene_a] & alt[:, gene_b] & joint_tested).sum())
        if both < MIN_BOTH:
            continue

        p_value, direction, expected = discover_pair_test(
            pi[joint_tested, gene_a], pi[joint_tested, gene_b], both)

        a_only = int((alt[:, gene_a] & ~alt[:, gene_b] & joint_tested).sum())
        b_only = int((~alt[:, gene_a] & alt[:, gene_b] & joint_tested).sum())
        neither = n_tested - both - a_only - b_only
        # Haldane-Anscombe-style +0.5 smoothing so fold_enrichment stays finite.
        fold = (both + 0.5) / (expected + 0.5)

        rows.append({
            "Cancer_Type": cancer_type, "Gene_A": genes[gene_a], "Gene_B": genes[gene_b],
            "n_tested_both": n_tested, "n_both_altered": both,
            "n_A_only": a_only, "n_B_only": b_only, "n_neither": neither,
            "expected_both": expected, "fold_enrichment": fold,
            "log2_fold_enrichment": np.log2(fold), "direction": direction, "p_value": p_value,
        })
    return pd.DataFrame(rows)

## 5. Benjamini-Hochberg FDR correction

Each cancer type ran its own batch of pairwise tests, so each cancer
type's p-values are corrected separately (a q-value in NSCLC shouldn't be
influenced by how many tests Breast Cancer happened to run).

In [ ]:
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    n = len(pvals)
    order = np.argsort(pvals)
    ranks = np.empty(n, dtype=int)
    ranks[order] = np.arange(1, n + 1)
    q = pvals * n / ranks
    q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
    q_final = np.empty(n)
    q_final[order] = np.clip(q_sorted, 0, 1)
    return q_final

## 6. Run across all qualifying cancer types

Two runs, using the two gene lists Phase 1 built:
1. **Main run** -- each cancer type uses its *own* coverage-qualified gene
   list (`consensus_genes_per_cancer_type.parquet`). This is the primary,
   most sensitive result: within a given cancer type, only genes that were
   both frequently altered AND reliably tested (>=80% of that cancer
   type's own patients, >=100 tested) enter the pairwise tests -- fixing
   the earlier gap where gene inclusion had no coverage floor at all.
2. **Global-gene run** -- every cancer type uses the *same* 190-gene list
   (`consensus_genes_global.parquet`), for a second, smaller table that's
   safe to directly compare *across* cancer types (Aim 1 asks for
   co-occurrence "within and across cancer types" -- this run is what
   makes the "across" comparison fair, since every cancer type is tested
   against an identical, consistently-covered gene set).

In [ ]:
consensus_per_ct = pd.read_parquet(PROCESSED_DIR / "consensus_genes_per_cancer_type.parquet")
genes_by_ct = consensus_per_ct.groupby("CANCER_TYPE")["Hugo_Symbol"].apply(set).to_dict()
global_genes = set(pd.read_parquet(PROCESSED_DIR / "consensus_genes_global.parquet")["Hugo_Symbol"])

# --- Main run: per-cancer-type coverage-qualified gene lists ---
all_results = []
# Main run is restricted to CNA-tested samples (see build_matrices) so that
# "altered" = "mutation OR copy-number change" is a claim every sample in the
# table could actually have supported.
cna_tested_counts = clinical[clinical["CNA_TESTED"]]["CANCER_TYPE"].value_counts()
main_cancer_types = [ct for ct in qualifying_cancer_types
                     if cna_tested_counts.get(ct, 0) >= MIN_SAMPLES_PER_CANCER_TYPE]
print(f"{len(main_cancer_types)} of {len(qualifying_cancer_types)} cancer types keep "
      f">= {MIN_SAMPLES_PER_CANCER_TYPE} samples once CNA-untested samples are excluded")

for ct in tqdm(main_cancer_types, desc="main (per-cancer-type genes)"):
    res = test_cancer_type(ct, genes_by_ct.get(ct, set()), require_cna_tested=True)
    if not res.empty:
        res["q_value"] = bh_fdr(res["p_value"].to_numpy())
        all_results.append(res)

comutation_pairs = pd.concat(all_results, ignore_index=True)
print("main run:", comutation_pairs.shape)
n_sig = (comutation_pairs["q_value"] < 0.05).sum()
print(f"{n_sig:,} of {len(comutation_pairs):,} pairs significant at q < 0.05 "
      f"({n_sig / len(comutation_pairs):.1%})")
comutation_pairs.head()

In [ ]:
# --- Global-gene run: same 190-gene list for every cancer type, for
# direct cross-cancer-type comparison ---
all_results_global = []
for ct in tqdm(main_cancer_types, desc="global (same 190 genes everywhere)"):
    res = test_cancer_type(ct, global_genes, require_cna_tested=True)
    if not res.empty:
        res["q_value"] = bh_fdr(res["p_value"].to_numpy())
        all_results_global.append(res)

comutation_pairs_global = pd.concat(all_results_global, ignore_index=True)
comutation_pairs_global.to_parquet(PROCESSED_DIR / "comutation_pairs_global_genes.parquet", index=False)
print("global-gene run:", comutation_pairs_global.shape)
n_sig_global = (comutation_pairs_global["q_value"] < 0.05).sum()
print(f"{n_sig_global:,} of {len(comutation_pairs_global):,} pairs significant at q < 0.05 "
      f"({n_sig_global / len(comutation_pairs_global):.1%})")
comutation_pairs_global.head()

### 6b. Mechanism-specific tests: which alteration types carry the signal

The runs above score a gene as altered if it carries **either** a mutation or a
copy-number change, which cannot distinguish "these two genes are both mutated"
from "one is mutated and the other is amplified". Those are different biological
claims, and they can point in opposite directions for the same pair.

So each pair is also tested four ways, mutation and copy-number treated
separately on each side:

| test | question |
|---|---|
| A-SNV x B-SNV | do point mutations in A and B co-occur? |
| A-SNV x B-CNV | does mutating A go with copy-number change in B? |
| A-CNV x B-SNV | the reverse -- asymmetric, so both are needed |
| A-CNV x B-CNV | do the copy-number events co-occur? |

Each test uses the same DISCOVER-style rate-adjusted null as the main run
(Section 4), not Fisher's exact test -- with one change: a patient's SNV
burden and CNV burden are different biological quantities (point-mutation
hypermutation vs. chromosomal instability aren't the same patient property),
so each mechanism gets its **own** rate fit (`fit_gene_sample_rates` run
separately on the SNV-only and CNV-only alteration matrices), not one shared
model. Each combination gets its own BH-FDR family (a q-value in the SNV-SNV
family shouldn't depend on how many CNV-CNV pairs happened to be testable).
All four share one denominator -- the CNA-tested samples of that cancer type
-- so the four results for a pair are directly comparable to each other.

The cross-mechanism tests are the ones that earn their place: `MDM2`
amplification vs `TP53` mutation is strongly mutually exclusive, and **no**
single-mechanism matrix can see it, because it needs one gene's copy-number
state compared against the other gene's mutation status.

In [ ]:
MECHANISMS = [("SNV", "SNV"), ("SNV", "CNV"), ("CNV", "SNV"), ("CNV", "CNV")]
MECH_MIN_ELIGIBLE = 50     # skip a pair entirely if the tested-both population is tiny
MECH_MIN_BOTH = 10         # skip a combo if fewer than this many samples carry both --
                           # at 5, ~22% of significant results rested on <10 patients and
                           # some OR>300 hits sat on n=6; 10 removes that tail

snv_events = alterations.loc[alterations["Alteration_Type"] == "MUT", ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
cnv_events = alterations.loc[alterations["Alteration_Type"] == "CNV", ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()


def mechanism_tests_for_cancer_type(cancer_type: str, allowed_genes: set[str]) -> pd.DataFrame:
    """Four mechanism-specific rate-adjusted tests per gene pair.

    Each side uses the testability mask that matches its own mechanism: the
    SNV side uses the mutation target list, the CNV side uses where
    copy-number calls actually exist (Phase 1 Section 2b). Using one blended
    mask for both -- as this did before -- puts samples into the CNV-side
    denominator whose panel never called copy number for that gene.
    """
    built = build_matrices(cancer_type, allowed_genes, require_cna_tested=True)
    if built is None:
        return pd.DataFrame()
    _, mut_cov, cna_cov, genes, n_ct_samples = built

    ct_samples = clinical.loc[clinical["CANCER_TYPE"] == cancer_type, ["SAMPLE_ID", "CNA_TESTED"]]
    index = ct_samples.loc[ct_samples["CNA_TESTED"], "SAMPLE_ID"].drop_duplicates()

    def event_matrix(events):
        sub = events[events["Sample_ID"].isin(index) & events["Hugo_Symbol"].isin(genes)]
        return (sub.assign(v=True)
                   .pivot(index="Sample_ID", columns="Hugo_Symbol", values="v")
                   .reindex(index=index, columns=genes).fillna(False).astype(bool)).to_numpy()

    M = {"SNV": event_matrix(snv_events), "CNV": event_matrix(cnv_events)}
    COV = {"SNV": mut_cov, "CNV": cna_cov}
    # Each mechanism's rate model is fit on its OWN testability mask -- a
    # patient's point-mutation burden and copy-number burden are different
    # biological quantities, measured on different gene sets.
    PI = {mech: 1 - np.exp(-fit_gene_sample_rates(M[mech], COV[mech])) for mech in M}

    rows = []
    for gene_a, gene_b in itertools.combinations(range(len(genes)), 2):
        for mech_a, mech_b in MECHANISMS:
            joint_tested = COV[mech_a][:, gene_a] & COV[mech_b][:, gene_b]
            n_tested = int(joint_tested.sum())
            if n_tested < MECH_MIN_ELIGIBLE or n_tested / n_ct_samples < JOINT_COVERAGE_FRACTION:
                continue
            a = M[mech_a][:, gene_a][joint_tested]
            b = M[mech_b][:, gene_b][joint_tested]
            both = int((a & b).sum())
            if both < MECH_MIN_BOTH:
                continue
            p_value, direction, expected = discover_pair_test(
                PI[mech_a][joint_tested, gene_a], PI[mech_b][joint_tested, gene_b], both)
            a_only, b_only = int((a & ~b).sum()), int((~a & b).sum())
            fold = (both + 0.5) / (expected + 0.5)
            rows.append({
                "Cancer_Type": cancer_type, "Gene_A": genes[gene_a], "Gene_B": genes[gene_b],
                "mech_A": mech_a, "mech_B": mech_b, "n_tested_both": n_tested,
                "n_both_altered": both, "n_A_only": a_only, "n_B_only": b_only,
                "n_neither": n_tested - both - a_only - b_only,
                "expected_both": expected, "fold_enrichment": fold,
                "log2_fold_enrichment": np.log2(fold), "direction": direction, "p_value": p_value,
            })
    return pd.DataFrame(rows)


mech_frames = []
for ct in tqdm(main_cancer_types, desc="mechanism-specific"):
    res = mechanism_tests_for_cancer_type(ct, genes_by_ct.get(ct, set()))
    if not res.empty:
        mech_frames.append(res)

mechanism_pairs = pd.concat(mech_frames, ignore_index=True)
# Own BH family per (cancer type, mechanism combo).
mechanism_pairs["q_value"] = np.nan
for _, grp in mechanism_pairs.groupby(["Cancer_Type", "mech_A", "mech_B"]):
    mechanism_pairs.loc[grp.index, "q_value"] = bh_fdr(grp["p_value"].to_numpy())

mechanism_pairs["combo"] = mechanism_pairs["mech_A"] + "-" + mechanism_pairs["mech_B"]
mechanism_pairs.to_parquet(PROCESSED_DIR / "mechanism_pairs.parquet", index=False)

print(f"{len(mechanism_pairs):,} mechanism-specific tests, "
      f"{int((mechanism_pairs['q_value'] < 0.05).sum()):,} significant")
print(mechanism_pairs.groupby("combo").agg(tested=("p_value", "size"),
                                           significant=("q_value", lambda s: int((s < 0.05).sum()))))

In [ ]:
# Cross-mechanism hits -- invisible to any single-mechanism matrix
sig_mech = mechanism_pairs[mechanism_pairs["q_value"] < 0.05]
cross = sig_mech[sig_mech["mech_A"] != sig_mech["mech_B"]]
print(f"significant cross-mechanism results: {len(cross):,}\n")
print("Strongest mutual exclusivity:")
print(cross.nsmallest(10, "fold_enrichment")[
    ["Cancer_Type", "Gene_A", "mech_A", "Gene_B", "mech_B", "n_tested_both",
     "n_both_altered", "expected_both", "fold_enrichment", "q_value"]].to_string(index=False))

# Pairs whose direction flips depending on which mechanisms are compared --
# exactly what the blended matrix averages away.
flip = sig_mech.copy()
flip["pair"] = [tuple(sorted([a, b])) for a, b in zip(flip["Gene_A"], flip["Gene_B"])]
conflicts = [(ct, pr, g) for (ct, pr), g in flip.groupby(["Cancer_Type", "pair"])
             if (g["direction"] == "co-occurring").any() and (g["direction"] == "exclusive").any()]
print(f"\n{len(conflicts)} pairs significant in BOTH directions depending on mechanism "
      f"(of {flip['pair'].nunique():,} significant pairs)")
for ct, pr, g in conflicts[:10]:
    print(f"  {ct:26s} {pr[0]}-{pr[1]:9s} " +
          " | ".join(f"{r.combo} fold={r.fold_enrichment:.2f} ({r.direction})" for r in g.itertuples()))

### 6c. Hypermutator robustness: the rest of the answer to "half seems high"

Section 4 fixed one confound (per-patient alteration burden) and the
significant fraction fell 72.3% -> 49.6%. It did not fall below "about
half", so the burden question was re-opened from a different angle:
**does the pooled rate reproduce within strata?**

Stratifying by alteration burden (top decile within each cancer type vs. the
rest) says no, and not subtly:

| | pooled | non-hypermutated (90% of patients) | hypermutated (10%) |
|---|---|---|---|
| Colorectal Cancer | 2,026 sig (77.2%) | 314 sig (**24.0%**) | 1,665 sig (65.6%), 99.7% co-occurring |
| Melanoma | 2,372 sig (41.2%) | 319 sig (**11.7%**) | 899 sig (20.0%), 100% co-occurring |

The pooled figure is reproduced in neither stratum, in both cancer types --
so this was never a Colorectal quirk (the earlier CIN-vs-MSI guess in this
notebook's Discussion was directionally right but far too narrow).

**Why Section 4's rate model doesn't already absorb this.** It gives each
patient one scalar rate multiplier, which handles *"this patient has more
alterations"*. Hypermutators differ in more than volume: MMR-deficient
tumours hit coding repeats, UV-driven melanomas hit particular sequence
contexts. Different mutational processes have different **gene
preferences**, which is correlated structure across genes -- and no
per-patient scalar can remove correlated structure. This is confounding by
population structure, and the standard handling in cancer genomics is to
stratify rather than pool.

**What's done about it.** The hypermutated decile isn't deleted -- their
biology is real and is arguably the more interesting half. Instead every
pair the pooled table reports is re-tested inside the non-hypermutated
stratum, and flagged:

- `nonhyper_q_value`, `nonhyper_direction` -- the re-test
- `robust_to_hypermutators` -- significant in **both**, same direction

**Result: 3,154 of 12,677 pooled-significant pairs (24.9%) are robust** --
12.3% of the 25,540 tested. The direction balance keeps improving as each
confound is removed:

| | co-occurring : exclusive |
|---|---|
| Fisher's exact test | 94.4% : 5.6% |
| + rate-adjusted test (Section 4) | 83.0% : 17.0% |
| + hypermutator-robust (this section) | **67.6% : 32.4%** |

Known biology is untouched by the filter: *KRAS*/*EGFR* and *KRAS*/*BRAF* in
NSCLC and *KRAS*/*BRAF* in colorectal are all `robust_to_hypermutators =
True`, and the strongest robust exclusivities are textbook -- *MDM2*/*TP53*
in soft tissue sarcoma (15 observed vs 312.6 expected, fold 0.05),
*CDK4*/*TP53*, *EGFR*/*KRAS*, *FGFR3*/*RB1* in bladder.

**So the headline to report is 3,154 robust pairs, not 12,677.** "Half of
tested pairs are significant" was the right thing to be suspicious of; the
honest number after removing both confounds is **12.3% of tested pairs**.

**Caveat on the cutoff**: hypermutation is proxied by the top decile of
distinct altered genes *within each cancer type*. Panels differ in size, so
a mutations/Mb threshold isn't available here; a within-cancer-type
percentile is transparent and panel-agnostic, but the 90th percentile is a
knob, not a biological constant. Thresholds per cancer type are saved to
`hypermutator_thresholds.parquet`. A true MSI/POLE annotation would be
better if it can be obtained.

In [ ]:
HYPERMUTATOR_PERCENTILE = 0.90  # provisional -- see caveat above

nonhyper_frames, burden_summary = [], []
for ct, ct_pairs in tqdm(comutation_pairs.groupby("Cancer_Type"), desc="hypermutator re-test"):
    ct_all = clinical.loc[(clinical["CANCER_TYPE"] == ct) & (clinical["CNA_TESTED"]), "SAMPLE_ID"]
    burden = (alterations[alterations["Sample_ID"].isin(set(ct_all))]
              .groupby("Sample_ID")["Hugo_Symbol"].nunique().reindex(ct_all, fill_value=0))
    cutoff = burden.quantile(HYPERMUTATOR_PERCENTILE)
    keep = set(burden[burden <= cutoff].index)
    burden_summary.append({"Cancer_Type": ct, "burden_cutoff": float(cutoff),
                            "n_total": int(len(burden)), "n_non_hypermutated": int(len(keep))})

    ct_samples = (clinical.loc[clinical["SAMPLE_ID"].isin(keep), ["SAMPLE_ID", "SEQ_ASSAY_ID"]]
                  .drop_duplicates("SAMPLE_ID").set_index("SAMPLE_ID"))
    n_ct = len(ct_samples)
    if n_ct < MIN_SAMPLES_PER_CANCER_TYPE:
        continue

    # Re-test exactly the pairs the pooled table reports -- like-for-like.
    genes = sorted(set(ct_pairs["Gene_A"]) | set(ct_pairs["Gene_B"]))
    sub_alt = alterations.loc[alterations["CANCER_TYPE"] == ct, ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
    sub_alt = sub_alt[sub_alt["Sample_ID"].isin(ct_samples.index) & sub_alt["Hugo_Symbol"].isin(genes)]
    alt = (sub_alt.assign(a=True).pivot(index="Sample_ID", columns="Hugo_Symbol", values="a")
           .reindex(index=ct_samples.index, columns=genes).fillna(False).astype(bool)).to_numpy()
    tested = (_coverage_mask(panel_coverage, ct_samples, genes)
              & _coverage_mask(cna_panel_coverage, ct_samples, genes))
    pi = 1 - np.exp(-fit_gene_sample_rates(alt, tested))
    gi = {g: k for k, g in enumerate(genes)}

    rows = []
    for a_gene, b_gene in zip(ct_pairs["Gene_A"], ct_pairs["Gene_B"]):
        i, j = gi[a_gene], gi[b_gene]
        joint = tested[:, i] & tested[:, j]
        nt = int(joint.sum())
        if nt < MIN_GENE_ALT_ABS or nt / n_ct < JOINT_COVERAGE_FRACTION:
            continue
        both = int((alt[:, i] & alt[:, j] & joint).sum())
        if both < MIN_BOTH:
            continue
        p, d, e = discover_pair_test(pi[joint, i], pi[joint, j], both)
        rows.append({"Cancer_Type": ct, "Gene_A": a_gene, "Gene_B": b_gene,
                     "nonhyper_n_tested_both": nt, "nonhyper_n_both_altered": both,
                     "nonhyper_expected_both": e, "nonhyper_direction": d, "nonhyper_p_value": p})
    if rows:
        df = pd.DataFrame(rows)
        df["nonhyper_q_value"] = bh_fdr(df["nonhyper_p_value"].to_numpy())
        nonhyper_frames.append(df)

nonhyper = pd.concat(nonhyper_frames, ignore_index=True)
comutation_pairs = comutation_pairs.merge(nonhyper, on=["Cancer_Type", "Gene_A", "Gene_B"], how="left")
comutation_pairs["robust_to_hypermutators"] = (
    (comutation_pairs["q_value"] < 0.05)
    & (comutation_pairs["nonhyper_q_value"] < 0.05)
    & (comutation_pairs["direction"] == comutation_pairs["nonhyper_direction"])
)
pd.DataFrame(burden_summary).to_parquet(PROCESSED_DIR / "hypermutator_thresholds.parquet", index=False)

_sig = comutation_pairs[comutation_pairs["q_value"] < 0.05]
_rob = comutation_pairs[comutation_pairs["robust_to_hypermutators"]]
print(f"pooled significant: {len(_sig):,}")
print(f"robust to hypermutators: {len(_rob):,} ({len(_rob)/len(_sig):.1%} of pooled significant, "
      f"{len(_rob)/len(comutation_pairs):.1%} of all tested)")
print(f"direction -- pooled: {(_sig['direction']=='co-occurring').mean():.1%} co-occurring | "
      f"robust: {(_rob['direction']=='co-occurring').mean():.1%} co-occurring")

### 6a. Mechanism annotation: MUT/CNV breakdown + genomic proximity

Every "altered" flag so far blends mutations and CNVs together (confirmed:
some pairs, like the 11q13/8p12 amplicon story, are ~97% CNV-driven -- a
physical/structural event, not two genes functionally cooperating). This
section adds, for every pair already tested:

1. **Mechanism breakdown** -- of the co-altered samples, how many were
   `MUT+MUT`, `CNV+CNV`, or mixed
2. **Genomic proximity** -- are the two genes physically close on the same
   chromosome (a single copy-number event could mechanically hit both,
   regardless of any real functional relationship)?

Neither of these filters anything out -- they're annotations added to help
interpret which findings look like genuine gene-gene interactions vs.
likely structural artifacts of genome geography.

In [ ]:
def build_gene_alteration_type_map(cancer_type: str, genes: list[str]) -> dict[str, dict[str, set]]:
    """For each gene, map Sample_ID -> set of Alteration_Type ('MUT'/'CNV') present."""
    ct_alt = alterations[(alterations["CANCER_TYPE"] == cancer_type) & (alterations["Hugo_Symbol"].isin(genes))]
    return {
        gene: grp.groupby("Sample_ID")["Alteration_Type"].apply(set).to_dict()
        for gene, grp in ct_alt.groupby("Hugo_Symbol", observed=True)
    }


def classify_pair_mechanism(map_a: dict, map_b: dict) -> dict:
    common = set(map_a) & set(map_b)
    counts = {"n_CNV_CNV": 0, "n_MUT_MUT": 0, "n_mixed": 0}
    for s in common:
        ta, tb = map_a[s], map_b[s]
        if "CNV" in ta and "CNV" in tb:
            counts["n_CNV_CNV"] += 1
        elif "MUT" in ta and "MUT" in tb:
            counts["n_MUT_MUT"] += 1
        else:
            counts["n_mixed"] += 1
    return counts


mechanism_rows = []
for cancer_type, ct_pairs in tqdm(comutation_pairs.groupby("Cancer_Type"), desc="mechanism annotation"):
    genes_here = sorted(set(ct_pairs["Gene_A"]) | set(ct_pairs["Gene_B"]))
    gene_maps = build_gene_alteration_type_map(cancer_type, genes_here)
    for row in ct_pairs.itertuples(index=False):
        counts = classify_pair_mechanism(gene_maps.get(row.Gene_A, {}), gene_maps.get(row.Gene_B, {}))
        mechanism_rows.append({"Cancer_Type": cancer_type, "Gene_A": row.Gene_A, "Gene_B": row.Gene_B, **counts})

mechanism_df = pd.DataFrame(mechanism_rows)
comutation_pairs = comutation_pairs.merge(mechanism_df, on=["Cancer_Type", "Gene_A", "Gene_B"], how="left")
comutation_pairs["frac_CNV_CNV"] = comutation_pairs["n_CNV_CNV"] / comutation_pairs["n_both_altered"].replace(0, pd.NA)
print(comutation_pairs[["Cancer_Type", "Gene_A", "Gene_B", "n_both_altered", "n_CNV_CNV", "n_MUT_MUT", "n_mixed", "frac_CNV_CNV"]].head())

**Data-quality finding**: building this genomic-distance check surfaced a
real issue in `genomic_information.txt` -- 33 of 19,605 genes (0.17%,
mostly immunoglobulin/TCR genes, known for complex, repetitive genomic
regions prone to mismapping) have implausible coordinate spans from bad
exon rows. E.g. `ZNF703` computed to a ~97 Mb span, when no real human
gene exceeds ~2.5 Mb (`DMD`, the largest, is ~2.4 Mb). Fixed by excluding
genes with an implausible span (>3 Mb) from the distance lookup entirely,
rather than trusting corrupted coordinates -- `genomic_distance_bp()`
already returns `None` for any gene missing from the footprint, so these
33 genes are just treated as "position unknown" going forward.

In [ ]:
# Genomic footprint per gene, collapsed from exon-level rows to one
# (chromosome, min start, max end) span per gene -- same source used for
# panel coverage in Phase 1.
gene_coords = pd.read_csv(
    RAW_DIR / "genomic_information.txt", sep="\t",
    usecols=["Hugo_Symbol", "Chromosome", "Start_Position", "End_Position"],
)
gene_footprint = gene_coords.groupby("Hugo_Symbol").agg(
    Chromosome=("Chromosome", "first"), Start=("Start_Position", "min"), End=("End_Position", "max"),
)
# See markdown note above -- excludes genes with corrupted coordinate spans.
MAX_PLAUSIBLE_GENE_SPAN_BP = 3_000_000
gene_footprint = gene_footprint[(gene_footprint["End"] - gene_footprint["Start"]) <= MAX_PLAUSIBLE_GENE_SPAN_BP]

PROXIMITY_THRESHOLD_BP = 10_000_000  # 10 Mb -- typical amplicon/deletion span; provisional, adjustable


def genomic_distance_bp(gene_a: str, gene_b: str):
    if gene_a not in gene_footprint.index or gene_b not in gene_footprint.index:
        return None
    fa, fb = gene_footprint.loc[gene_a], gene_footprint.loc[gene_b]
    if str(fa["Chromosome"]) != str(fb["Chromosome"]):
        return None  # different chromosome -- not proximate, by definition
    if fa["End"] >= fb["Start"] and fb["End"] >= fa["Start"]:
        return 0  # overlapping
    return int(max(fa["Start"], fb["Start"]) - min(fa["End"], fb["End"]))


comutation_pairs["genomic_distance_bp"] = [
    genomic_distance_bp(a, b) for a, b in zip(comutation_pairs["Gene_A"], comutation_pairs["Gene_B"])
]
comutation_pairs["likely_physical_proximity"] = (
    comutation_pairs["genomic_distance_bp"].notna()
    & (comutation_pairs["genomic_distance_bp"] <= PROXIMITY_THRESHOLD_BP)
)

n_proximate = comutation_pairs["likely_physical_proximity"].sum()
print(f"{n_proximate:,} of {len(comutation_pairs):,} pairs are on the same chromosome within {PROXIMITY_THRESHOLD_BP/1e6:.0f} Mb")
print(f"Of significant pairs (q<0.05): {comutation_pairs.loc[comutation_pairs['q_value']<0.05, 'likely_physical_proximity'].mean():.1%} flagged as likely physical proximity")

## 7. Across cancers: pan-cancer recurrence

Everything so far tests co-occurrence **within** one cancer type at a time
-- Aim 1 also asks for patterns **across** cancers. Pooling raw samples
from every cancer type into one big test was considered and rejected: it
would be dominated by whichever cancer type has the most samples (NSCLC:
39,553 vs. e.g. Medulloblastoma: 105) -- a classic Simpson's-paradox risk,
not a real cross-cancer signal.

Instead, this reuses the already-correct per-cancer-type results: for each
gene pair, count how many cancer types it's significant in, and whether
the direction (co-occurring vs. exclusive) is **consistent** across them.
A pair recurring with the same direction across many independent cancer
types is real evidence of a cross-cancer pattern; a pair that's
significant in one cancer type but flips direction in another reflects
real tissue-specific biology, not noise -- both are informative, but only
the first is a genuine "across cancers" finding.

In [ ]:
PAN_CANCER_MIN_CANCER_TYPES = 5  # provisional -- how many cancer types before calling a pair "pan-cancer"

sig = comutation_pairs[comutation_pairs["q_value"] < 0.05].copy()
sig["pair_key"] = list(zip(sig["Gene_A"].where(sig["Gene_A"] < sig["Gene_B"], sig["Gene_B"]),
                            sig["Gene_B"].where(sig["Gene_A"] < sig["Gene_B"], sig["Gene_A"])))
# 'direction' (co-occurring / exclusive) already comes straight from the
# DISCOVER-style test (Section 4) -- no need to re-derive it from an odds ratio.

pan_cancer = sig.groupby("pair_key").agg(
    n_cancer_types_significant=("Cancer_Type", "nunique"),
    cancer_types=("Cancer_Type", lambda s: sorted(set(s))),
    directions=("direction", lambda s: sorted(set(s))),
).reset_index()
pan_cancer["Gene_A"] = pan_cancer["pair_key"].apply(lambda t: t[0])
pan_cancer["Gene_B"] = pan_cancer["pair_key"].apply(lambda t: t[1])
pan_cancer["direction_consistent"] = pan_cancer["directions"].apply(len) == 1
pan_cancer["direction"] = pan_cancer["directions"].apply(lambda d: d[0] if len(d) == 1 else "mixed")
pan_cancer = pan_cancer.drop(columns="pair_key").sort_values("n_cancer_types_significant", ascending=False)

pan_cancer["is_pan_cancer_candidate"] = (
    (pan_cancer["n_cancer_types_significant"] >= PAN_CANCER_MIN_CANCER_TYPES) & pan_cancer["direction_consistent"]
)

n_candidates = pan_cancer["is_pan_cancer_candidate"].sum()
print(f"{n_candidates} pan-cancer candidate pairs (significant in >= {PAN_CANCER_MIN_CANCER_TYPES} cancer types, consistent direction)")
print()
print("Top 10 by recurrence:")
print(pan_cancer[["Gene_A", "Gene_B", "n_cancer_types_significant", "direction", "direction_consistent"]].head(10).to_string())

pan_cancer.to_parquet(PROCESSED_DIR / "pan_cancer_pairs.parquet", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
top_pan = pan_cancer.sort_values("n_cancer_types_significant", ascending=False).head(20)
labels = [f"{a}-{b}" for a, b in zip(top_pan["Gene_A"], top_pan["Gene_B"])]
colors = top_pan["direction_consistent"].map({True: "#55A868", False: "#C44E52"})

ax.barh(labels[::-1], top_pan["n_cancer_types_significant"].values[::-1], color=colors.values[::-1])
ax.set_xlabel("Number of cancer types significant in")
ax.set_title("Top 20 gene pairs by pan-cancer recurrence")
ax.legend(
    handles=[Patch(color="#55A868", label="consistent direction"), Patch(color="#C44E52", label="direction flips")],
    fontsize=8, loc="lower right",
)
plt.tight_layout()
plt.show()

## 8. Sanity check against known biology

A few pairs with well-established mutual exclusivity in the literature --
if the pipeline is working, these should show odds ratio < 1 (mutually
exclusive) with a low q-value.

In [ ]:
known_pairs = [
    ("Non-Small Cell Lung Cancer", "KRAS", "EGFR"),
    ("Colorectal Cancer", "KRAS", "BRAF"),
    ("Breast Cancer", "PIK3CA", "PTEN"),
]

for cancer_type, gene_a, gene_b in known_pairs:
    match = comutation_pairs[
        (comutation_pairs["Cancer_Type"] == cancer_type)
        & (comutation_pairs["Gene_A"].isin([gene_a, gene_b]))
        & (comutation_pairs["Gene_B"].isin([gene_a, gene_b]))
    ]
    print(cancer_type, gene_a, "vs", gene_b)
    print(match[["n_both_altered", "expected_both", "n_A_only", "n_B_only", "n_neither",
                 "fold_enrichment", "direction", "p_value", "q_value"]])
    print()

## 9. Most significant enrichments and exclusivities overall

In [ ]:
significant = comutation_pairs[comutation_pairs["q_value"] < 0.05]
print(f"{len(significant):,} of {len(comutation_pairs):,} pairs significant at q < 0.05")

print("\nTop co-occurring pairs (highest fold enrichment):")
print(significant.sort_values("log2_fold_enrichment", ascending=False)
      [["Cancer_Type", "Gene_A", "Gene_B", "n_both_altered", "expected_both", "fold_enrichment", "q_value"]].head(10))

print("\nTop mutually exclusive pairs (lowest fold enrichment):")
print(significant.sort_values("log2_fold_enrichment")
      [["Cancer_Type", "Gene_A", "Gene_B", "n_both_altered", "expected_both", "fold_enrichment", "q_value"]].head(10))

## 10. Discussion: 11q13 / 8p12 co-amplification -- a known breast cancer pattern

The two things worth flagging from the results above:

1. **Top co-occurring pairs are dominated by `FGF19`/`FGF4`/`FGF3`/`CCND1`**,
   often with extreme (even infinite) odds ratios. This is expected, not a
   bug: these 4 genes sit physically next to each other on chromosome
   11q13. When that region gets amplified, all of them get duplicated
   together as one physical event -- not because they cooperate
   biologically, but because they're neighbors on the same piece of DNA.

2. **11q13 genes co-occur with 8p12 genes (`FGFR1`, `WHSC1L1`) specifically
   in Breast Cancer**, and nowhere else with the same strength -- a
   recognized, subtype-specific finding in the literature (common in
   ER+/luminal B breast cancers), not a generic genomic artifact. Checked
   below.

In [ ]:
genes_11q13 = ["CCND1", "FGF3", "FGF4", "FGF19"]
genes_8p12 = ["FGFR1", "WHSC1L1", "ZNF703", "NSD3"]

mask = (
    (comutation_pairs["Gene_A"].isin(genes_11q13) & comutation_pairs["Gene_B"].isin(genes_8p12))
    | (comutation_pairs["Gene_A"].isin(genes_8p12) & comutation_pairs["Gene_B"].isin(genes_11q13))
)
cross_region = comutation_pairs[mask].sort_values("q_value")
print(f"{len(cross_region)} cross 11q13-8p12 pairs tested")
cross_region[["Cancer_Type", "Gene_A", "Gene_B", "n_both_altered", "expected_both", "fold_enrichment", "direction", "q_value"]]

**Result:** all 8 cross-region pairs tested in Breast Cancer come back
significantly co-occurring (fold enrichment ~1.66-1.78 over the rate-adjusted
expectation, q as low as ~1.7e-48). The same gene pairs in Bladder Cancer
show no significant relationship at all (q all > 0.05, fold ~0.83-1.04 --
essentially the rate-adjusted null). Endometrial Cancer is more interesting
than it looked under Fisher's test: both tested pairs there are significant,
but with fold enrichment *below* 1 (~0.62-0.63) -- i.e. these regions are
mildly **mutually exclusive** in Endometrial Cancer, not weakly co-occurring
as the old Fisher-based numbers suggested. So the pipeline still shows this
co-amplification as breast-cancer-specific (matching the literature), and
the rate-adjusted test additionally surfaces a real, different pattern in
Endometrial Cancer that the old test had labelled as merely "weaker" rather
than "different direction" -- a second independent sanity check (beyond the
KRAS/EGFR-style pairwise exclusivity checks in Section 8) that the pipeline
reproduces real, specific cancer genomics rather than generic statistical
noise.

## 11. Save output

In [ ]:
comutation_pairs.to_parquet(PROCESSED_DIR / "comutation_pairs.parquet", index=False)
print(f"saved {len(comutation_pairs):,} rows to {PROCESSED_DIR / 'comutation_pairs.parquet'}")

## 12. Visualize

A volcano plot of every pair tested (effect size vs. significance,
the standard way to look at a large batch of statistical tests at once),
with the 3 known-biology pairs from the sanity check highlighted, plus
which cancer types produced the most significant pairs. (The pan-cancer
recurrence chart is right after Section 7, next to the result it
visualizes, rather than bundled in here.)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

not_sig = comutation_pairs[comutation_pairs["q_value"] >= 0.05]
sig = comutation_pairs[comutation_pairs["q_value"] < 0.05]

ax.scatter(not_sig["log2_fold_enrichment"], -np.log10(not_sig["q_value"].clip(lower=1e-300)),
           s=4, alpha=0.15, color="lightgray", label="not significant")
ax.scatter(sig["log2_fold_enrichment"], -np.log10(sig["q_value"].clip(lower=1e-300)),
           s=4, alpha=0.3, color="#4C72B0", label="significant (q < 0.05)")

for cancer_type, gene_a, gene_b in known_pairs:
    row = comutation_pairs[
        (comutation_pairs["Cancer_Type"] == cancer_type)
        & (comutation_pairs["Gene_A"].isin([gene_a, gene_b]))
        & (comutation_pairs["Gene_B"].isin([gene_a, gene_b]))
    ]
    if not row.empty:
        r = row.iloc[0]
        y = -np.log10(max(r["q_value"], 1e-300))
        ax.scatter([r["log2_fold_enrichment"]], [y], s=60, color="red", zorder=5, edgecolor="black")
        ax.annotate(f"{gene_a}-{gene_b}\n({cancer_type})", (r["log2_fold_enrichment"], y),
                    fontsize=7, xytext=(5, 5), textcoords="offset points")

ax.axvline(0, color="black", linewidth=0.5, linestyle="--")
ax.set_xlabel("log2(fold enrichment over rate-adjusted expectation)   <- mutually exclusive | co-occurring ->")
ax.set_ylabel("-log10(q-value)")
ax.set_title(f"All {len(comutation_pairs):,} gene pairs tested, across {comutation_pairs['Cancer_Type'].nunique()} cancer types")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sig_per_ct = significant["Cancer_Type"].value_counts().head(15)
ax.barh(sig_per_ct.index[::-1], sig_per_ct.values[::-1], color="#55A868")
ax.set_xlabel("Number of significant pairs (q < 0.05)")
ax.set_title("Top 15 cancer types by significant pair count")
plt.tight_layout()
plt.show()

## Discussion

**Validation:** the pipeline reproduces well-known cancer biology without
being told to look for it -- *KRAS*/*EGFR* and *KRAS*/*BRAF* mutual
exclusivity, *EGFR*/*IDH1* mutual exclusivity in glioma (the two branches of
glioma biology), and the 11q13/8p12 co-amplification specific to breast
cancer. That's the main evidence the method measures something real.

**Why Fisher's exact test was replaced.** Jason's question after seeing
early results: *"Are you using a Fisher's exact test to evaluate significant
co-occurrence or mutual exclusivity? Half being significant sounds quite
high as I would imagine many gene pairs have very few samples. Also you
should apply multiple testing correction."* BH-FDR correction (Section 5)
was already in place, but checking the "half seems high" concern directly
turned up a real, specific, fixable cause:

- Per-sample alteration burden is wildly skewed within a cancer type (e.g.
  Melanoma: median 6 genes altered per patient, 90th pct 26, max 494).
  Fisher's exact test assumes every patient has the same baseline chance of
  any given gene being altered -- badly wrong here.
- Checked directly: under Fisher's test, 90-94% of significant pairs were
  "co-occurring" and only 6-9% "exclusive," most lopsided in exactly the
  cancer types already flagged elsewhere as inherently hypermutated (Skin
  Cancer Non-Melanoma, Endometrial, Melanoma). A burden confound inflates
  apparent co-occurrence specifically (a hypermutated patient's dozens of
  alterations make *every* gene pair on their panel look "co-occurring"),
  while leaving genuine exclusivity comparatively untouched.
- Fix: a DISCOVER-style test (Canisius et al. 2016, *Genome Biology*) --
  fit a per-gene, per-patient rate model (Section 4a) instead of assuming
  one pooled probability, then test each pair against the Poisson-Binomial
  null those rates imply. Controlled comparison (identical gene lists,
  identical floors, only the test differs): significant pairs dropped from
  72.3% to 49.6%, and the co-occurring:exclusive split improved from
  94.4%:5.6% to 82.9%:17.1%. Known biology got *clearer*, not weaker --
  *KRAS*/*EGFR* in NSCLC: 173 observed vs. 1,425.5 expected co-mutations
  under the adjusted null (q≈0).

**What we discovered along the way:**

1. **Which patients were tested matters as much as what was found.** Once
   patients with no copy-number data were excluded from copy-number
   comparisons (Phase 1's `CNA_TESTED`), several rare cancer types dropped
   out entirely because too few of their patients had ever been tested.
   Their old results were really mutation-only results wearing a "combined"
   label.

2. **Lumping mutations and copy-number changes together hides the most
   interesting patterns.** Testing each gene pair four ways (mutation vs
   mutation, mutation vs copy-number, and the reverse, copy-number vs
   copy-number) revealed **1,637 significant cross-mechanism results that no
   single-mechanism table can see** -- and the strongest ones are textbook:
   - *MDM2* gained vs *TP53* mutated: almost never together (fold enrichment
     0.13-0.19 of the rate-adjusted expectation, in bladder **and** sarcoma
     independently). Extra *MDM2* already switches off the TP53 protein, so
     the tumour gains nothing by also breaking the gene.
   - *EGFR* mutated vs *IDH1* mutated in glioma (fold=0.16, and *EGFR*
     *gained* vs *IDH1* mutated, fold=0.11): the two branches of glioma
     biology, almost never in the same tumour, visible on both the
     SNV-SNV and CNV-SNV tables independently.
   - 285 pairs point in **opposite directions** depending on mechanism --
     e.g. in Bladder Cancer, a *CDKN2A* point mutation and a *CDKN2B*
     copy-number change avoid each other (fold=0.30), but *CDKN2A* and
     *CDKN2B* copy-number changes strongly co-occur (fold=3.42) -- expected,
     since they're physical neighbours and a single deletion event can take
     both out together, but a point mutation in one doesn't need its
     neighbour to also be lost. The blended table averages both into
     nothing.

3. **The copy-number "bystander" label earns its place** (a Phase 1 result,
   unaffected by the test-statistic change here): dropping direction-
   inconsistent calls removed known bystander artifacts (e.g. *BRIP1*
   "amplified" riding the 17q amplicon next to the gene that actually
   matters) while leaving textbook results untouched. Genes OncoKB has
   never reviewed are deliberately **kept**: dropping them too would mean
   only ever rediscovering already-known cancer genes.

**Result:** main table 25,540 gene pairs across 52 cancer types, 12,677
significant (49.6%) -- but **3,154 (12.3% of tested) survive the
hypermutator-robustness check** in Section 6c, which is the number worth
reporting. Mechanism-specific table 27,723 tests, 14,006 significant
(50.5%), of which 1,658 cross-mechanism. 191 pan-cancer candidate pairs
(significant in >=5 cancer types with a consistent direction).

**Two further corrections since the test-statistic change** (both found by
checking rather than assuming):

1. **Testability is alteration-type-specific** (Phase 1 Section 2b).
   `panel_gene_coverage.parquet` is each panel's *mutation* target list;
   where copy-number calls actually exist is a different thing, and they
   disagree for 16.6% of (panel, gene) combinations -- ~199,000
   (sample, gene) cells that a single blended mask scored as "tested, no
   copy-number change" when copy number was never assessed there. Fixed by
   giving copy number its own coverage table. Evidence the fix is surgical:
   SNV-SNV mechanism results are bit-identical before and after (20,936
   tested / 11,242 significant), while every copy-number-involving
   combination moved.

2. **Hypermutator population structure** (Section 6c). The pooled
   significance rate reproduces in neither burden stratum: Colorectal
   77.2% pooled -> 24.0% among the non-hypermutated 90%; Melanoma 41.2% ->
   11.7%. The rate model's per-patient scalar absorbs alteration *volume*
   but not the *gene preferences* of a different mutational process, so
   hypermutators inject correlated structure it cannot remove. Handled by
   re-testing every pair in the non-hypermutated stratum and flagging
   `robust_to_hypermutators`, rather than deleting those patients.

**Limitations:**
- Restricting to copy-number-tested patients costs ~27% of patients per
  comparison and several small cancer types. A mutation-only run on all
  patients would recover those; not yet built.
- **New trade-off from the joint-coverage gate + `MIN_BOTH>=10` floor**:
  the main run now covers 52 cancer types, down from 58 under the old rule
  -- a handful of small cancer types no longer have *any* pair that clears
  both the joint-coverage bar and the minimum-co-occurrence floor. This is
  a real cost of demanding enough evidence per pair, not a bug, but worth
  knowing: statistical rigor per pair traded against breadth of cancer-type
  coverage.
- ~~Colorectal Cancer's significance rate barely moved under the
  rate-adjusted test; possibly its CIN-vs-MSI subtype split.~~
  **Investigated and resolved (Section 6c)**: the guess was directionally
  right but far too narrow. Stratifying by alteration burden shows the
  pooled rate reproduces in neither stratum, in Melanoma as much as in
  Colorectal -- this is general hypermutator population structure, not a
  Colorectal quirk. Now handled by the `robust_to_hypermutators` flag.
- The hypermutation cutoff (top decile of altered genes within each cancer
  type) is a provisional proxy. Panels differ in size so mutations/Mb isn't
  available; a real MSI/POLE annotation would be better if obtainable.
- Cross-mechanism **co-occurrence** hits can reflect a shared histological
  subtype rather than a real interaction (*PIK3CA* amp + *TP53* mut in lung
  is mostly squamous histology). Exclusivity hits look far more robust.
  Stratifying by `CANCER_TYPE_DETAILED` separates the two cleanly (see
  notebook 06).
- Physically adjacent genes (*CDKN2A*/*CDKN2B*) can still co-occur for a
  purely structural reason on the CNV-CNV table specifically; the
  genomic-proximity flag marks these, and the mechanism breakdown (point 2
  above) shows this doesn't hold across every mechanism combination for
  the same pair.
- The rate model (Section 4a) fits gene and patient effects multiplicatively
  (`lam = gene_rate x sample_rate`); it doesn't model gene-gene covariates
  beyond that (e.g. pathway membership). A richer model is possible but
  wasn't necessary to answer the specific concern this section addresses.

**Open questions:** whether the hypermutation proxy should be replaced by a
real MSI/POLE annotation; whether to build the mutation-only run to recover
the CNA-untested cancer types; the fusion analysis for lung cancer; whether
downstream phases should consume the robust subset (3,154 pairs) or the full
significant set (12,677) -- Phase 3's clustering will look very different
depending on which.

## Next steps (Phase 3)

Use `comutation_pairs.parquet` (the significant, q < 0.05 subset) as a
graph -- genes as nodes, significant co-occurring pairs as edges -- and
run community detection (e.g. Louvain) to find 3+ gene clusters, rather
than just pairs. See `PLAN.md` Phase 3.